In [ ]:
!pip install apache_beam

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.0/152.0 kB 9.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.3/46.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.5/261.5 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31

In [ ]:
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions
from datetime import datetime
from pyspark.sql import SparkSession
import pandas as pd


In [ ]:
# Initialize Spark Session
spark = SparkSession.builder.appName("IncrementalLoad").getOrCreate()

In [ ]:
#testing Scenario 1
class FilterLatest(beam.DoFn):
    """Filter newly inserted records based on 'erdate'."""
    def __init__(self, last_load_date):
        self.last_load_date = datetime.strptime(last_load_date, "%Y-%m-%d")

    def process(self, record):
        erdate = datetime.strptime(record[6], "%Y-%m-%d")  # 'erdate' is at index 6
        if erdate > self.last_load_date:
            yield record

class FilterUpdated(beam.DoFn):
    """Filter updated records based on 'udate'."""
    def __init__(self, last_load_date):
        self.last_load_date = datetime.strptime(last_load_date, "%Y-%m-%d")

    def process(self, record):
        udate = datetime.strptime(record[7], "%Y-%m-%d") if record[7] and record[7] != "NULL" else None  # 'udate' is at index 7
        if udate and udate > self.last_load_date:
            yield record

# Define the last load date (current date - 1)
LAST_LOAD_DATE = (datetime.today() - timedelta(days=1)).strftime("%Y-%m-%d")

# Beam Pipeline
pipeline_options = PipelineOptions()
with beam.Pipeline(options=pipeline_options) as pipeline:

    # Read CSV and parse lines into lists
    source_data = (
        pipeline
        | "Read CSV File" >> beam.io.ReadFromText('/content/sample - Sheet1 (1).csv', skip_header_lines=1)
        | "Parse CSV to List" >> beam.Map(lambda line: line.split(','))
    )

    # Filter Newly Inserted and Updated Records
    latest_records = source_data | "Filter Latest Records" >> beam.ParDo(FilterLatest(LAST_LOAD_DATE))
    updated_records = source_data | "Filter Updated Records" >> beam.ParDo(FilterUpdated(LAST_LOAD_DATE))

    # Convert to lists for PySpark DataFrame creationupdated_records | "Collect Updated Data" >> beam.combiners.ToList()

    latest_list = latest_records | "Collect Latest Data" >> beam.combiners.ToList()
    updated_list =
    # Store data for conversion to PySpark
    def store_latest(data):
        global latest_data
        latest_data = data

    def store_updated(data):
        global updated_data
        updated_data = data

    latest_list | "Store Latest Records" >> beam.Map(store_latest)
    updated_list | "Store Updated Records" >> beam.Map(store_updated)




In [ ]:
# Define Schema
schema = ["id", "first_name", "last_name", "age", "salary", "department", "erdate", "udate"]

In [ ]:
# Convert Lists to PySpark DataFrames
latest_df = spark.createDataFrame(pd.DataFrame(latest_data, columns=schema))
updated_df = spark.createDataFrame(pd.DataFrame(updated_data, columns=schema))

In [ ]:
# Show DataFrames
print("Latest Inserted Records:")
latest_df.show()


Latest Inserted Records:
+---+----------+---------+---+------+----------+----------+----------+
| id|first_name|last_name|age|salary|department|    erdate|     udate|
+---+----------+---------+---+------+----------+----------+----------+
|  4|     Emily|    Davis| 28| 58000| Marketing|2025-03-10|          |
|  5|    Daniel| Martinez| 40| 90000|        IT|2025-03-11|2025-03-11|
|  6|    Sophia|    Lopez| 27| 62000|        HR|2025-03-12|          |
|  7|     James| Gonzalez| 45|110000|   Finance|2025-03-13|2025-03-13|
|  8|    Olivia|   Wilson| 32| 70000| Marketing|2025-03-14|          |
|  9|   William| Anderson| 29| 63000|        IT|2025-03-15|2025-03-15|
| 10|       Ava|   Thomas| 26| 59000|        HR|2025-03-16|2025-03-16|
| 11| Alexander|   Taylor| 38| 85000|   Finance|2025-03-17|          |
| 12|       Mia|    Moore| 24| 54000| Marketing|2025-03-18|2025-03-18|
| 13|     Ethan|  Jackson| 31| 67000|        IT|2025-03-19|2025-03-19|
| 14| Charlotte|    White| 33| 72000|        HR|2025

In [ ]:
print("\nUpdated Records:")
updated_df.show()



Updated Records:
+---+----------+---------+---+------+----------+----------+----------+
| id|first_name|last_name|age|salary|department|    erdate|     udate|
+---+----------+---------+---+------+----------+----------+----------+
|  5|    Daniel| Martinez| 40| 90000|        IT|2025-03-11|2025-03-11|
|  7|     James| Gonzalez| 45|110000|   Finance|2025-03-13|2025-03-13|
|  9|   William| Anderson| 29| 63000|        IT|2025-03-15|2025-03-15|
| 10|       Ava|   Thomas| 26| 59000|        HR|2025-03-16|2025-03-16|
| 12|       Mia|    Moore| 24| 54000| Marketing|2025-03-18|2025-03-18|
| 13|     Ethan|  Jackson| 31| 67000|        IT|2025-03-19|2025-03-19|
| 15|  Benjamin|   Harris| 37| 78000|   Finance|2025-03-21|2025-03-21|
| 17|     Lucas| Robinson| 42| 98000|        IT|2025-03-23|2025-03-23|
| 19|     Mason|     Hall| 34| 74000|   Finance|2025-03-25|2025-03-25|
+---+----------+---------+---+------+----------+----------+----------+



In [ ]:
from datetime import datetime, timedelta

# Define LAST_LOAD_DATE as yesterday's date
LAST_LOAD_DATE = (datetime.today() - timedelta(days=1)).strftime("%Y-%m-%d")

print("Last Load Date:", LAST_LOAD_DATE)  # Just to verify


Last Load Date: 2025-03-09


In [ ]:
indexes = [i for i, x in enumerate(my_list) if x == element_to_find]

In [ ]:
#testing scenario2

import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions
from datetime import datetime, timedelta

class FilterLatest(beam.DoFn):
    """Filter newly inserted records based on 'erdate'."""
    def __init__(self, last_load_date):
        self.last_load_date = datetime.strptime(last_load_date, "%Y-%m-%d")

    def process(self, record, indexes):
        erdate_index = indexes[0]  # Get 'erdate' column index

        erdate = datetime.strptime(record[erdate_index], "%Y-%m-%d")
        if erdate > self.last_load_date:
            yield record


class FilterUpdated(beam.DoFn):
    """Filter updated records based on 'udate'."""
    def __init__(self, last_load_date):
        self.last_load_date = datetime.strptime(last_load_date, "%Y-%m-%d")

    def process(self, record, indexes):
        udate_index = indexes[1]  # Get 'udate' column index
        udate = datetime.strptime(record[udate_index], "%Y-%m-%d") if record[udate_index] and record[udate_index] != "NULL" else None
        if udate and udate > self.last_load_date:
            yield record


# Define the last load date (current date - 1)
LAST_LOAD_DATE = (datetime.today() - timedelta(days=1)).strftime("%Y-%m-%d")

pipeline_options = PipelineOptions()

with beam.Pipeline(options=pipeline_options) as pipeline:

    #  Read CSV file
    lines = pipeline | "Read CSV File" >> beam.io.ReadFromText('/content/sample - Sheet1 (1).csv')

    # Extract header (first row)
    header = (
        lines
        | "Extract Header" >> beam.combiners.ToList()
        | "Get First Row" >> beam.FlatMap(lambda rows: [rows[0].split(',')])  # Ensure single-element list
    )

    # Extract data rows (skip header)
    data_rows = lines | "Skip Header" >> beam.Filter(lambda line: not line.startswith("id"))

    # Convert data rows to lists
    parsed_data = data_rows | "Parse CSV to List" >> beam.Map(lambda line: line.split(','))

    #  Find indexes dynamically
    indexes = header | "Extract Indexes" >> beam.Map(lambda h: (h.index("erdate"), h.index("udate")))

    #  Use indexes as a side input in `ParDo`
    latest_records = (
        parsed_data
        | "Filter Latest Records" >> beam.ParDo(FilterLatest(LAST_LOAD_DATE), indexes=beam.pvalue.AsSingleton(indexes))
    )

    updated_records = (
        parsed_data
        | "Filter Updated Records" >> beam.ParDo(FilterUpdated(LAST_LOAD_DATE), indexes=beam.pvalue.AsSingleton(indexes))
    )

    # Convert to lists for PySpark DataFrame creation
    latest_list = latest_records | "Collect Latest Data" >> beam.combiners.ToList()
    updated_list = updated_records | "Collect Updated Data" >> beam.combiners.ToList()

    # Store data for conversion to PySpark
    def store_latest(data):
        global latest_data
        latest_data = data

    def store_updated(data):
        global updated_data
        updated_data = data


    latest_list | "Store Latest Records" >> beam.Map(store_latest)
    updated_list | "Store Updated Records" >> beam.Map(store_updated)


In [ ]:
print(latest_data)

[['4', 'Emily', 'Davis', '28', '58000', 'Marketing', '2025-03-10', ''], ['5', 'Daniel', 'Martinez', '40', '90000', 'IT', '2025-03-11', '2025-03-11'], ['6', 'Sophia', 'Lopez', '27', '62000', 'HR', '2025-03-12', ''], ['7', 'James', 'Gonzalez', '45', '110000', 'Finance', '2025-03-13', '2025-03-13'], ['8', 'Olivia', 'Wilson', '32', '70000', 'Marketing', '2025-03-14', ''], ['9', 'William', 'Anderson', '29', '63000', 'IT', '2025-03-15', '2025-03-15'], ['10', 'Ava', 'Thomas', '26', '59000', 'HR', '2025-03-16', '2025-03-16'], ['11', 'Alexander', 'Taylor', '38', '85000', 'Finance', '2025-03-17', ''], ['12', 'Mia', 'Moore', '24', '54000', 'Marketing', '2025-03-18', '2025-03-18'], ['13', 'Ethan', 'Jackson', '31', '67000', 'IT', '2025-03-19', '2025-03-19'], ['14', 'Charlotte', 'White', '33', '72000', 'HR', '2025-03-20', ''], ['15', 'Benjamin', 'Harris', '37', '78000', 'Finance', '2025-03-21', '2025-03-21'], ['16', 'Amelia', 'Clark', '29', '64000', 'Marketing', '2025-03-22', ''], ['17', 'Lucas', 'R

In [ ]:
print(updated_data)

[['5', 'Daniel', 'Martinez', '40', '90000', 'IT', '2025-03-11', '2025-03-11'], ['7', 'James', 'Gonzalez', '45', '110000', 'Finance', '2025-03-13', '2025-03-13'], ['9', 'William', 'Anderson', '29', '63000', 'IT', '2025-03-15', '2025-03-15'], ['10', 'Ava', 'Thomas', '26', '59000', 'HR', '2025-03-16', '2025-03-16'], ['12', 'Mia', 'Moore', '24', '54000', 'Marketing', '2025-03-18', '2025-03-18'], ['13', 'Ethan', 'Jackson', '31', '67000', 'IT', '2025-03-19', '2025-03-19'], ['15', 'Benjamin', 'Harris', '37', '78000', 'Finance', '2025-03-21', '2025-03-21'], ['17', 'Lucas', 'Robinson', '42', '98000', 'IT', '2025-03-23', '2025-03-23'], ['19', 'Mason', 'Hall', '34', '74000', 'Finance', '2025-03-25', '2025-03-25']]


**Dynamic the pipeline**

In [ ]:
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions
from datetime import datetime, timedelta

class FilterLatest(beam.DoFn):
    """Filter newly inserted records based on 'erdate'."""
    def __init__(self, last_load_date):
        self.last_load_date = datetime.strptime(last_load_date, "%Y-%m-%d")

    def process(self, record, indexes):
        erdate_index = indexes[0]  # Get 'erdate' column index
        erdate = datetime.strptime(record[erdate_index], "%Y-%m-%d")
        if erdate > self.last_load_date:
            yield record


class FilterUpdated(beam.DoFn):
    """Filter updated records based on 'udate'."""
    def __init__(self, last_load_date):
        self.last_load_date = datetime.strptime(last_load_date, "%Y-%m-%d")

    def process(self, record, indexes):
        udate_index = indexes[1]  # Get 'udate' column index

            udate = datetime.strptime(record[udate_index], "%Y-%m-%d") if record[udate_index] and record[udate_index] != "NULL" else None
            if udate and udate > self.last_load_date:
                yield record


# Define the last load date (current date - 1)
LAST_LOAD_DATE = (datetime.today() - timedelta(days=1)).strftime("%Y-%m-%d")

pipeline_options = PipelineOptions()

with beam.Pipeline(options=pipeline_options) as pipeline:

    #  Read CSV file
    lines = pipeline | "Read CSV File" >> beam.io.ReadFromText('/content/sample - Sheet1 (1).csv')

    # Extract header (first row)
    header = (
        lines
        | "Extract Header" >> beam.combiners.ToList()
        | "Get First Row" >> beam.FlatMap(lambda rows: [rows[0].split(',')])  # Ensure single-element list
    )

    # Extract data rows (skip header)
    data_rows = lines | "Skip Header" >> beam.Filter(lambda line: not line.startswith("id"))

    # Convert data rows to lists
    parsed_data = data_rows | "Parse CSV to List" >> beam.Map(lambda line: line.split(','))

    #  Find indexes dynamically
    indexes = header | "Extract Indexes" >> beam.Map(lambda h: (h.index("erdate"), h.index("udate")))

    #  Use indexes as a side input in `ParDo`
    latest_records = (
        parsed_data
        | "Filter Latest Records" >> beam.ParDo(FilterLatest(LAST_LOAD_DATE), indexes=beam.pvalue.AsSingleton(indexes))
    )

    updated_records = (
        parsed_data
        | "Filter Updated Records" >> beam.ParDo(FilterUpdated(LAST_LOAD_DATE), indexes=beam.pvalue.AsSingleton(indexes))
    )

    Schema_table =  header | "Get schema of the table" >> beam.combiners.ToList()
    # Convert to lists for PySpark DataFrame creation
    latest_list = latest_records | "Collect Latest Data" >> beam.combiners.ToList()
    updated_list = updated_records | "Collect Updated Data" >> beam.combiners.ToList()

    # Store data for conversion to PySpark
    def store_latest(data):
        global latest_data
        latest_data = data

    def store_updated(data):
        global updated_data
        updated_data = data

    def header_to_schema(Schemas):
        global Schema_table
        Schema_table = Schemas

    header | "Convert Header to Schema" >> beam.Map(header_to_schema)

    latest_list | "Store Latest Records" >> beam.Map(store_latest)
    updated_list | "Store Updated Records" >> beam.Map(store_updated)


In [ ]:
print(Schema_table)

['id', 'first_name', 'last_name', 'age', 'salary', 'department', 'erdate', 'udate']


In [ ]:
print(updated_data)

[['5', 'Daniel', 'Martinez', '40', '90000', 'IT', '2025-03-11', '2025-03-11'], ['7', 'James', 'Gonzalez', '45', '110000', 'Finance', '2025-03-13', '2025-03-13'], ['9', 'William', 'Anderson', '29', '63000', 'IT', '2025-03-15', '2025-03-15'], ['10', 'Ava', 'Thomas', '26', '59000', 'HR', '2025-03-16', '2025-03-16'], ['12', 'Mia', 'Moore', '24', '54000', 'Marketing', '2025-03-18', '2025-03-18'], ['13', 'Ethan', 'Jackson', '31', '67000', 'IT', '2025-03-19', '2025-03-19'], ['15', 'Benjamin', 'Harris', '37', '78000', 'Finance', '2025-03-21', '2025-03-21'], ['17', 'Lucas', 'Robinson', '42', '98000', 'IT', '2025-03-23', '2025-03-23'], ['19', 'Mason', 'Hall', '34', '74000', 'Finance', '2025-03-25', '2025-03-25']]


In [ ]:
updated_records =  spark.createDataFrame(updated_data, Schema_table)

In [ ]:
updated_records.show()

+---+----------+---------+---+------+----------+----------+----------+
| id|first_name|last_name|age|salary|department|    erdate|     udate|
+---+----------+---------+---+------+----------+----------+----------+
|  5|    Daniel| Martinez| 40| 90000|        IT|2025-03-11|2025-03-11|
|  7|     James| Gonzalez| 45|110000|   Finance|2025-03-13|2025-03-13|
|  9|   William| Anderson| 29| 63000|        IT|2025-03-15|2025-03-15|
| 10|       Ava|   Thomas| 26| 59000|        HR|2025-03-16|2025-03-16|
| 12|       Mia|    Moore| 24| 54000| Marketing|2025-03-18|2025-03-18|
| 13|     Ethan|  Jackson| 31| 67000|        IT|2025-03-19|2025-03-19|
| 15|  Benjamin|   Harris| 37| 78000|   Finance|2025-03-21|2025-03-21|
| 17|     Lucas| Robinson| 42| 98000|        IT|2025-03-23|2025-03-23|
| 19|     Mason|     Hall| 34| 74000|   Finance|2025-03-25|2025-03-25|
+---+----------+---------+---+------+----------+----------+----------+



In [ ]:
print(latest_data)

[['4', 'Emily', 'Davis', '28', '58000', 'Marketing', '2025-03-10', ''], ['5', 'Daniel', 'Martinez', '40', '90000', 'IT', '2025-03-11', '2025-03-11'], ['6', 'Sophia', 'Lopez', '27', '62000', 'HR', '2025-03-12', ''], ['7', 'James', 'Gonzalez', '45', '110000', 'Finance', '2025-03-13', '2025-03-13'], ['8', 'Olivia', 'Wilson', '32', '70000', 'Marketing', '2025-03-14', ''], ['9', 'William', 'Anderson', '29', '63000', 'IT', '2025-03-15', '2025-03-15'], ['10', 'Ava', 'Thomas', '26', '59000', 'HR', '2025-03-16', '2025-03-16'], ['11', 'Alexander', 'Taylor', '38', '85000', 'Finance', '2025-03-17', ''], ['12', 'Mia', 'Moore', '24', '54000', 'Marketing', '2025-03-18', '2025-03-18'], ['13', 'Ethan', 'Jackson', '31', '67000', 'IT', '2025-03-19', '2025-03-19'], ['14', 'Charlotte', 'White', '33', '72000', 'HR', '2025-03-20', ''], ['15', 'Benjamin', 'Harris', '37', '78000', 'Finance', '2025-03-21', '2025-03-21'], ['16', 'Amelia', 'Clark', '29', '64000', 'Marketing', '2025-03-22', ''], ['17', 'Lucas', 'R

In [ ]:
inserted_records =  spark.createDataFrame(latest_data, Schema_table)

In [ ]:
inserted_records.show()

+---+----------+---------+---+------+----------+----------+----------+
| id|first_name|last_name|age|salary|department|    erdate|     udate|
+---+----------+---------+---+------+----------+----------+----------+
|  4|     Emily|    Davis| 28| 58000| Marketing|2025-03-10|          |
|  5|    Daniel| Martinez| 40| 90000|        IT|2025-03-11|2025-03-11|
|  6|    Sophia|    Lopez| 27| 62000|        HR|2025-03-12|          |
|  7|     James| Gonzalez| 45|110000|   Finance|2025-03-13|2025-03-13|
|  8|    Olivia|   Wilson| 32| 70000| Marketing|2025-03-14|          |
|  9|   William| Anderson| 29| 63000|        IT|2025-03-15|2025-03-15|
| 10|       Ava|   Thomas| 26| 59000|        HR|2025-03-16|2025-03-16|
| 11| Alexander|   Taylor| 38| 85000|   Finance|2025-03-17|          |
| 12|       Mia|    Moore| 24| 54000| Marketing|2025-03-18|2025-03-18|
| 13|     Ethan|  Jackson| 31| 67000|        IT|2025-03-19|2025-03-19|
| 14| Charlotte|    White| 33| 72000|        HR|2025-03-20|          |
| 15| 

In [ ]:
target_df =  spark.read.csv('/content/sample - Sheet1 (2).csv'
                            , header=True, inferSchema=True)

In [ ]:
target_df.show()

+---+----------+---------+---+------+----------+----------+----------+
| id|first_name|last_name|age|salary|department|    erdate|     udate|
+---+----------+---------+---+------+----------+----------+----------+
|  1|      John|      Doe| 30| 60000|        IT|2025-03-07|2025-03-07|
|  2|      Jane|    Smith| 25| 55000|        HR|2025-03-08|      NULL|
|  3|   Michael|  Johnson| 35| 75000|   Finance|2025-03-09|2025-03-09|
|  4|     Emily|    Davis| 28| 58000| Marketing|2025-03-10|      NULL|
|  5|    Daniel| Martinez| 40| 90000|        IT|2025-03-11|2025-03-11|
|  6|    Sophia|    Lopez| 27| 62000|        HR|2025-03-12|      NULL|
|  7|     James| Gonzalez| 45|110000|   Finance|2025-03-13|2025-03-13|
|  8|    Olivia|   Wilson| 32| 70000| Marketing|2025-03-14|      NULL|
|  9|   William| Anderson| 29| 63000|        IT|2025-03-15|2025-03-15|
| 10|       Ava|   Thomas| 26| 59000|        HR|2025-03-16|2025-03-16|
| 11| Alexander|   Taylor| 38| 85000|   Finance|2025-03-17|      NULL|
| 12| 

In [ ]:
# Perform Full Outer Join
merged_df = target_df.alias("target").join(
    inserted_records.alias("source"),
    on="id",
    how="outer"
)


In [ ]:
merged_df.show()

+---+----------+---------+---+------+----------+----------+----------+----------+---------+----+------+----------+----------+----------+
| id|first_name|last_name|age|salary|department|    erdate|     udate|first_name|last_name| age|salary|department|    erdate|     udate|
+---+----------+---------+---+------+----------+----------+----------+----------+---------+----+------+----------+----------+----------+
|  1|      John|      Doe| 30| 60000|        IT|2025-03-07|2025-03-07|      NULL|     NULL|NULL|  NULL|      NULL|      NULL|      NULL|
|  2|      Jane|    Smith| 25| 55000|        HR|2025-03-08|      NULL|      NULL|     NULL|NULL|  NULL|      NULL|      NULL|      NULL|
|  3|   Michael|  Johnson| 35| 75000|   Finance|2025-03-09|2025-03-09|      NULL|     NULL|NULL|  NULL|      NULL|      NULL|      NULL|
|  4|     Emily|    Davis| 28| 58000| Marketing|2025-03-10|      NULL|     Emily|    Davis|  28| 58000| Marketing|2025-03-10|          |
|  5|    Daniel| Martinez| 40| 90000|    

In [ ]:
from pyspark.sql.functions import *

In [ ]:
# Apply Merge Logic
final_df = merged_df.select(
    col("id"),
    when(col("source.first_name").isNotNull(), col("source.first_name")).otherwise(col("target.first_name")).alias("first_name"),
    when(col("source.last_name").isNotNull(), col("source.last_name")).otherwise(col("target.last_name")).alias("last_name"),
    when(col("source.age").isNotNull(), col("source.age")).otherwise(col("target.age")).alias("age"),
    when(col("source.salary").isNotNull(), col("source.salary")).otherwise(col("target.salary")).alias("salary"),
    when(col("source.department").isNotNull(), col("source.department")).otherwise(col("target.department")).alias("department"),
    when(col("source.erdate").isNotNull(), col("source.erdate")).otherwise(col("target.erdate")).alias("erdate"),
    when(col("source.udate").isNotNull(), col("source.udate")).otherwise(col("target.udate")).alias("udate")
)

In [ ]:
final_df.show()

+---+----------+---------+---+------+----------+----------+----------+
| id|first_name|last_name|age|salary|department|    erdate|     udate|
+---+----------+---------+---+------+----------+----------+----------+
|  1|      John|      Doe| 30| 60000|        IT|2025-03-07|2025-03-07|
|  2|      Jane|    Smith| 25| 55000|        HR|2025-03-08|      NULL|
|  3|   Michael|  Johnson| 35| 75000|   Finance|2025-03-09|2025-03-09|
|  4|     Emily|    Davis| 28| 58000| Marketing|2025-03-10|      NULL|
|  5|    Daniel| Martinez| 40| 90000|        IT|2025-03-11|2025-03-11|
|  6|    Sophia|    Lopez| 27| 62000|        HR|2025-03-12|      NULL|
|  7|     James| Gonzalez| 45|110000|   Finance|2025-03-13|2025-03-13|
|  8|    Olivia|   Wilson| 32| 70000| Marketing|2025-03-14|      NULL|
|  9|   William| Anderson| 29| 63000|        IT|2025-03-15|2025-03-15|
| 10|       Ava|   Thomas| 26| 59000|        HR|2025-03-16|2025-03-16|
| 11| Alexander|   Taylor| 38| 85000|   Finance|2025-03-17|      NULL|
| 12| 

In [ ]:
# Perform Full Outer Join
merged_df = final_df.alias("target").join(
    updated_records.alias("source"),
    on="id",
    how="outer"
)


In [ ]:
merged_df.orderBy('id').show()

+---+----------+---------+---+------+----------+----------+----------+----------+---------+----+------+----------+----------+----------+
| id|first_name|last_name|age|salary|department|    erdate|     udate|first_name|last_name| age|salary|department|    erdate|     udate|
+---+----------+---------+---+------+----------+----------+----------+----------+---------+----+------+----------+----------+----------+
|  1|      John|      Doe| 30| 60000|        IT|2025-03-07|2025-03-07|      NULL|     NULL|NULL|  NULL|      NULL|      NULL|      NULL|
| 10|       Ava|   Thomas| 26| 59000|        HR|2025-03-16|2025-03-16|       Ava|   Thomas|  26| 59000|        HR|2025-03-16|2025-03-16|
| 11| Alexander|   Taylor| 38| 85000|   Finance|2025-03-17|      NULL|      NULL|     NULL|NULL|  NULL|      NULL|      NULL|      NULL|
| 12|       Mia|    Moore| 24| 54000| Marketing|2025-03-18|2025-03-18|       Mia|    Moore|  24| 54000| Marketing|2025-03-18|2025-03-18|
| 13|     Ethan|  Jackson| 31| 67000|    

In [ ]:
# Apply Merge Logic
final_df = merged_df.select(
    col("id"),
    when(col("source.first_name").isNotNull(), col("source.first_name")).otherwise(col("target.first_name")).alias("first_name"),
    when(col("source.last_name").isNotNull(), col("source.last_name")).otherwise(col("target.last_name")).alias("last_name"),
    when(col("source.age").isNotNull(), col("source.age")).otherwise(col("target.age")).alias("age"),
    when(col("source.salary").isNotNull(), col("source.salary")).otherwise(col("target.salary")).alias("salary"),
    when(col("source.department").isNotNull(), col("source.department")).otherwise(col("target.department")).alias("department"),
    when(col("source.erdate").isNotNull(), col("source.erdate")).otherwise(col("target.erdate")).alias("erdate"),
    when(col("source.udate").isNotNull(), col("source.udate")).otherwise(col("target.udate")).alias("udate")
)

In [ ]:
final_df.orderBy("id").show()

+---+----------+---------+---+------+----------+----------+----------+
| id|first_name|last_name|age|salary|department|    erdate|     udate|
+---+----------+---------+---+------+----------+----------+----------+
|  1|      John|      Doe| 30| 60000|        IT|2025-03-07|2025-03-07|
| 10|       Ava|   Thomas| 26| 59000|        HR|2025-03-16|2025-03-16|
| 11| Alexander|   Taylor| 38| 85000|   Finance|2025-03-17|      NULL|
| 12|       Mia|    Moore| 24| 54000| Marketing|2025-03-18|2025-03-18|
| 13|     Ethan|  Jackson| 31| 67000|        IT|2025-03-19|2025-03-19|
| 14| Charlotte|    White| 33| 72000|        HR|2025-03-20|      NULL|
| 15|  Benjamin|   Harris| 37| 78000|   Finance|2025-03-21|2025-03-21|
| 16|    Amelia|    Clark| 29| 64000| Marketing|2025-03-22|      NULL|
| 17|     Lucas| Robinson| 42| 98000|        IT|2025-03-23|2025-03-23|
| 18|    Harper|   Walker| 30| 66000|        HR|2025-03-24|      NULL|
| 19|     Mason|     Hall| 34| 74000|   Finance|2025-03-25|2025-03-25|
|  2| 

**Dynamic the pipeline2**

In [ ]:
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions
from datetime import datetime, timedelta

file_list = ['sample - Sheet1 (1).csv', 'sample - Sheet1 (2).csv']


class FilterLatest(beam.DoFn):
    """Filter newly inserted records based on 'erdate'."""
    def __init__(self, last_load_date):
        self.last_load_date = datetime.strptime(last_load_date, "%Y-%m-%d")

    def process(self, record, indexes):
        erdate_index = indexes[0]  # Get 'erdate' column index

        erdate = datetime.strptime(record[erdate_index], "%Y-%m-%d")
        if erdate > self.last_load_date:
            yield record


class FilterUpdated(beam.DoFn):
    """Filter updated records based on 'udate'."""
    def __init__(self, last_load_date):
        self.last_load_date = datetime.strptime(last_load_date, "%Y-%m-%d")

    def process(self, record, indexes):
        udate_index = indexes[1]  # Get 'udate' column index

        udate = datetime.strptime(record[udate_index], "%Y-%m-%d") if record[udate_index] and record[udate_index] != "NULL" else None
        if udate and udate > self.last_load_date:
            yield record


# Define the last load date (current date - 1)
LAST_LOAD_DATE = (datetime.today() - timedelta(days=1)).strftime("%Y-%m-%d")

def process_file(file_name):

  pipeline_options = PipelineOptions()

  with beam.Pipeline(options=pipeline_options) as pipeline:

      #  Read CSV file
      lines = pipeline | "Read CSV File" >> beam.io.ReadFromText(f'/content/{file_name}')

      # Extract header (first row)
      header = (
          lines
          | "Extract Header" >> beam.combiners.ToList()
          | "Get First Row" >> beam.FlatMap(lambda rows: [rows[0].split(',')])  # Ensure single-element list
      )

      # Extract data rows (skip header)
      data_rows = lines | "Skip Header" >> beam.Filter(lambda line: not line.startswith("id"))

      # Convert data rows to lists
      parsed_data = data_rows | "Parse CSV to List" >> beam.Map(lambda line: line.split(','))

      #  Find indexes dynamically
      indexes = header | "Extract Indexes" >> beam.Map(lambda h: (h.index("erdate"), h.index("udate")))

      #  Use indexes as a side input in `ParDo`
      latest_records = (
          parsed_data
          | "Filter Latest Records" >> beam.ParDo(FilterLatest(LAST_LOAD_DATE), indexes=beam.pvalue.AsSingleton(indexes))
      )

      updated_records = (
          parsed_data
          | "Filter Updated Records" >> beam.ParDo(FilterUpdated(LAST_LOAD_DATE), indexes=beam.pvalue.AsSingleton(indexes))
      )

      Schema_table =  header | "Get schema of the table" >> beam.combiners.ToList()
      # Convert to lists for PySpark DataFrame creation
      latest_list = latest_records | "Collect Latest Data" >> beam.combiners.ToList()
      updated_list = updated_records | "Collect Updated Data" >> beam.combiners.ToList()

      # Store data for conversion to PySpark
      def store_latest(data):
          global latest_data
          latest_data = data

      def store_updated(data):
          global updated_data
          updated_data = data

      def header_to_schema(Schemas):
          global Schema_table
          Schema_table = Schemas

      header | "Convert Header to Schema" >> beam.Map(header_to_schema)

      latest_list | "Store Latest Records" >> beam.Map(store_latest)
      updated_list | "Store Updated Records" >> beam.Map(store_updated)


for file_name in file_list:
    process_file(file_name)

In [ ]:
print(latest_data)

[['4', 'Emily', 'Davis', '28', '58000', 'Marketing', '2025-03-10', ''], ['5', 'Daniel', 'Martinez', '40', '90000', 'IT', '2025-03-11', '2025-03-11'], ['6', 'Sophia', 'Lopez', '27', '62000', 'HR', '2025-03-12', ''], ['7', 'James', 'Gonzalez', '45', '110000', 'Finance', '2025-03-13', '2025-03-13'], ['8', 'Olivia', 'Wilson', '32', '70000', 'Marketing', '2025-03-14', ''], ['9', 'William', 'Anderson', '29', '63000', 'IT', '2025-03-15', '2025-03-15'], ['10', 'Ava', 'Thomas', '26', '59000', 'HR', '2025-03-16', '2025-03-16'], ['11', 'Alexander', 'Taylor', '38', '85000', 'Finance', '2025-03-17', ''], ['12', 'Mia', 'Moore', '24', '54000', 'Marketing', '2025-03-18', '2025-03-18'], ['13', 'Ethan', 'Jackson', '31', '67000', 'IT', '2025-03-19', '2025-03-19'], ['14', 'Charlotte', 'White', '33', '72000', 'HR', '2025-03-20', ''], ['15', 'Benjamin', 'Harris', '37', '78000', 'Finance', '2025-03-21', '2025-03-21'], ['16', 'Amelia', 'Clark', '29', '64000', 'Marketing', '2025-03-22', ''], ['17', 'Lucas', 'R

In [ ]:
print(Schema_table)

['id', 'first_name', 'last_name', 'age', 'salary', 'department', 'erdate', 'udate']


In [ ]:
df = spark.createDataFrame(updated_data, Schema_table)

In [ ]:
df.show()

+---+----------+---------+---+------+----------+----------+----------+
| id|first_name|last_name|age|salary|department|    erdate|     udate|
+---+----------+---------+---+------+----------+----------+----------+
|  5|    Daniel| Martinez| 40| 90000|        IT|2025-03-11|2025-03-11|
|  7|     James| Gonzalez| 45|110000|   Finance|2025-03-13|2025-03-13|
|  9|   William| Anderson| 29| 63000|        IT|2025-03-15|2025-03-15|
| 10|       Ava|   Thomas| 26| 59000|        HR|2025-03-16|2025-03-16|
| 12|       Mia|    Moore| 24| 54000| Marketing|2025-03-18|2025-03-18|
| 13|     Ethan|  Jackson| 31| 67000|        IT|2025-03-19|2025-03-19|
| 15|  Benjamin|   Harris| 37| 78000|   Finance|2025-03-21|2025-03-21|
| 17|     Lucas| Robinson| 42| 98000|        IT|2025-03-23|2025-03-23|
| 19|     Mason|     Hall| 34| 74000|   Finance|2025-03-25|2025-03-25|
+---+----------+---------+---+------+----------+----------+----------+



In [ ]:
#Overwriting the record in a list


import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions
from datetime import datetime, timedelta

# List of CSV files to process
file_list = ['sample - Sheet1 (1).csv', 'sample - Sheet1 (2).csv']

# Store the final results
latest_data_list = []
updated_data_list = []

class FilterLatest(beam.DoFn):
    """Filter newly inserted records based on 'erdate'."""
    def __init__(self, last_load_date):
        self.last_load_date = datetime.strptime(last_load_date, "%Y-%m-%d")

    def process(self, record, indexes):
        erdate_index = indexes[0]  # Get 'erdate' column index
        erdate = datetime.strptime(record[erdate_index], "%Y-%m-%d")
        if erdate > self.last_load_date:
            yield record

class FilterUpdated(beam.DoFn):
    """Filter updated records based on 'udate'."""
    def __init__(self, last_load_date):
        self.last_load_date = datetime.strptime(last_load_date, "%Y-%m-%d")

    def process(self, record, indexes):
        udate_index = indexes[1]  # Get 'udate' column index
        udate = datetime.strptime(record[udate_index], "%Y-%m-%d") if record[udate_index] and record[udate_index] != "NULL" else None
        if udate and udate > self.last_load_date:
            yield record

# Define the last load date (yesterday)
LAST_LOAD_DATE = (datetime.today() - timedelta(days=1)).strftime("%Y-%m-%d")

def set_latest_data(data):
    """Store latest records in a list."""
    global latest_data_list
    latest_data_list = data

def set_updated_data(data):
    """Store updated records in a list."""
    global updated_data_list
    updated_data_list = data

def process_file(file_name):
    pipeline_options = PipelineOptions()

    with beam.Pipeline(options=pipeline_options) as pipeline:
        # Read CSV file dynamically
        lines = pipeline | f"Read {file_name}" >> beam.io.ReadFromText(file_name)

        # Extract header correctly
        header = (
            lines
            | f"Extract Header {file_name}" >> beam.combiners.ToList()
            | f"Get First Row {file_name}" >> beam.Map(lambda rows: rows[0].split(',') if rows else [])
        )

        # Extract data rows (skip header)
        data_rows = lines | f"Filter Data {file_name}" >> beam.Filter(lambda line: not line.startswith("id"))

        # Convert data rows to lists
        parsed_data = data_rows | f"Parse CSV {file_name}" >> beam.Map(lambda line: line.split(','))

        # Find indexes dynamically
        indexes = header | f"Find Indexes {file_name}" >> beam.Map(lambda h: (h.index("erdate"), h.index("udate")))

        # Use indexes as a side input in `ParDo`
        latest_records = (
            parsed_data
            | f"Filter Latest {file_name}" >> beam.ParDo(FilterLatest(LAST_LOAD_DATE), indexes=beam.pvalue.AsSingleton(indexes))
        )

        updated_records = (
            parsed_data
            | f"Filter Updated {file_name}" >> beam.ParDo(FilterUpdated(LAST_LOAD_DATE), indexes=beam.pvalue.AsSingleton(indexes))
        )

        # Collect records into lists
        latest_list = latest_records | f"Collect Latest {file_name}" >> beam.combiners.ToList()
        updated_list = updated_records | f"Collect Updated {file_name}" >> beam.combiners.ToList()

        # Store results
        latest_list | f"Store Latest {file_name}" >> beam.Map(set_latest_data)
        updated_list | f"Store Updated {file_name}" >> beam.Map(set_updated_data)

        # Run pipeline
        result = pipeline.run()
        result.wait_until_finish()

# Process each file
for file_name in file_list:
    process_file(file_name)

# Print final results
print("Latest Records:", latest_data_list)
print("Updated Records:", updated_data_list)


Latest Records: [['4', 'Emily', 'Davis', '28', '58000', 'Marketing', '2025-03-10', ''], ['5', 'Daniel', 'Martinez', '40', '90000', 'IT', '2025-03-11', '2025-03-11'], ['6', 'Sophia', 'Lopez', '27', '62000', 'HR', '2025-03-12', ''], ['7', 'James', 'Gonzalez', '45', '110000', 'Finance', '2025-03-13', '2025-03-13'], ['8', 'Olivia', 'Wilson', '32', '70000', 'Marketing', '2025-03-14', ''], ['9', 'William', 'Anderson', '29', '63000', 'IT', '2025-03-15', '2025-03-15'], ['10', 'Ava', 'Thomas', '26', '59000', 'HR', '2025-03-16', '2025-03-16'], ['11', 'Alexander', 'Taylor', '38', '85000', 'Finance', '2025-03-17', ''], ['12', 'Mia', 'Moore', '24', '54000', 'Marketing', '2025-03-18', '2025-03-18'], ['13', 'Ethan', 'Jackson', '31', '67000', 'IT', '2025-03-19', '2025-03-19'], ['14', 'Charlotte', 'White', '33', '72000', 'HR', '2025-03-20', ''], ['15', 'Benjamin', 'Harris', '37', '78000', 'Finance', '2025-03-21', '2025-03-21'], ['16', 'Amelia', 'Clark', '29', '64000', 'Marketing', '2025-03-22', ''], ['

In [ ]:
df = spark.createDataFrame(latest_data_list, Schema_table)

In [ ]:
df.show()

+---+----------+---------+---+------+----------+----------+----------+
| id|first_name|last_name|age|salary|department|    erdate|     udate|
+---+----------+---------+---+------+----------+----------+----------+
|  4|     Emily|    Davis| 28| 58000| Marketing|2025-03-10|          |
|  5|    Daniel| Martinez| 40| 90000|        IT|2025-03-11|2025-03-11|
|  6|    Sophia|    Lopez| 27| 62000|        HR|2025-03-12|          |
|  7|     James| Gonzalez| 45|110000|   Finance|2025-03-13|2025-03-13|
|  8|    Olivia|   Wilson| 32| 70000| Marketing|2025-03-14|          |
|  9|   William| Anderson| 29| 63000|        IT|2025-03-15|2025-03-15|
| 10|       Ava|   Thomas| 26| 59000|        HR|2025-03-16|2025-03-16|
| 11| Alexander|   Taylor| 38| 85000|   Finance|2025-03-17|          |
| 12|       Mia|    Moore| 24| 54000| Marketing|2025-03-18|2025-03-18|
| 13|     Ethan|  Jackson| 31| 67000|        IT|2025-03-19|2025-03-19|
| 14| Charlotte|    White| 33| 72000|        HR|2025-03-20|          |
| 15| 

In [ ]:
df2 = spark.createDataFrame(updated_data_list, Schema_table)


In [ ]:
df2.show()

+---+----------+---------+---+------+----------+----------+----------+
| id|first_name|last_name|age|salary|department|    erdate|     udate|
+---+----------+---------+---+------+----------+----------+----------+
|  5|    Daniel| Martinez| 40| 90000|        IT|2025-03-11|2025-03-11|
|  7|     James| Gonzalez| 45|110000|   Finance|2025-03-13|2025-03-13|
|  9|   William| Anderson| 29| 63000|        IT|2025-03-15|2025-03-15|
| 10|       Ava|   Thomas| 26| 59000|        HR|2025-03-16|2025-03-16|
| 12|       Mia|    Moore| 24| 54000| Marketing|2025-03-18|2025-03-18|
| 13|     Ethan|  Jackson| 31| 67000|        IT|2025-03-19|2025-03-19|
| 15|  Benjamin|   Harris| 37| 78000|   Finance|2025-03-21|2025-03-21|
| 17|     Lucas| Robinson| 42| 98000|        IT|2025-03-23|2025-03-23|
| 19|     Mason|     Hall| 34| 74000|   Finance|2025-03-25|2025-03-25|
|  5|    Daniel| Martinez| 40| 90000|        IT|2025-03-11|2025-03-11|
|  7|     James| Gonzalez| 45|110000|   Finance|2025-03-13|2025-03-13|
|  9| 

In [ ]:
#appending the records
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions
from datetime import datetime, timedelta

# List of CSV files to process
file_list = ['sample - Sheet1 (1).csv', 'sample - Sheet1 (2).csv']

# Store the final results
latest_data_list = []
updated_data_list = []

class FilterLatest(beam.DoFn):
    """Filter newly inserted records based on 'erdate'."""
    def __init__(self, last_load_date):
        self.last_load_date = datetime.strptime(last_load_date, "%Y-%m-%d")

    def process(self, record, indexes):
        erdate_index = indexes[0]
        erdate = datetime.strptime(record[erdate_index], "%Y-%m-%d")
        if erdate > self.last_load_date:
            yield record

class FilterUpdated(beam.DoFn):
    """Filter updated records based on 'udate'."""
    def __init__(self, last_load_date):
        self.last_load_date = datetime.strptime(last_load_date, "%Y-%m-%d")

    def process(self, record, indexes):
        udate_index = indexes[1]
        udate = datetime.strptime(record[udate_index], "%Y-%m-%d") if record[udate_index] and record[udate_index] != "NULL" else None
        if udate and udate > self.last_load_date:
            yield record

# Define the last load date (yesterday)
LAST_LOAD_DATE = (datetime.today() - timedelta(days=1)).strftime("%Y-%m-%d")

def set_latest_data(data):
    """Append latest records to a list."""
    global latest_data_list
    latest_data_list =  data

def set_updated_data(data):
    """Append updated records to a list."""
    global updated_data_list
    updated_data_list =  data

def process_file(file_name):
    pipeline_options = PipelineOptions()

    with beam.Pipeline(options=pipeline_options) as pipeline:
        # Read CSV file dynamically
        lines = pipeline | f"Read {file_name}" >> beam.io.ReadFromText(file_name)

        # Extract header correctly
        header = (
            lines
            | f"Extract Header {file_name}" >> beam.combiners.ToList()
            | f"Get First Row {file_name}" >> beam.Map(lambda rows: rows[0].split(',') if rows else [])
        )

        # Extract data rows (skip header)
        data_rows = lines | f"Filter Data {file_name}" >> beam.Filter(lambda line: not line.startswith("id"))

        # Convert data rows to lists
        parsed_data = data_rows | f"Parse CSV {file_name}" >> beam.Map(lambda line: line.split(','))

        # Find indexes dynamically
        indexes = header | f"Find Indexes {file_name}" >> beam.Map(lambda h: (h.index("erdate"), h.index("udate")))

        # Use indexes as a side input in `ParDo`
        latest_records = (
            parsed_data
            | f"Filter Latest {file_name}" >> beam.ParDo(FilterLatest(LAST_LOAD_DATE), indexes=beam.pvalue.AsSingleton(indexes))
        )

        updated_records = (
            parsed_data
            | f"Filter Updated {file_name}" >> beam.ParDo(FilterUpdated(LAST_LOAD_DATE), indexes=beam.pvalue.AsSingleton(indexes))
        )

        # Collect records into lists
        latest_list = latest_records | f"Collect Latest {file_name}" >> beam.combiners.ToList()
        updated_list = updated_records | f"Collect Updated {file_name}" >> beam.combiners.ToList()

        # Store results
        latest_list | f"Store Latest {file_name}" >> beam.Map(set_latest_data)
        updated_list | f"Store Updated {file_name}" >> beam.Map(set_updated_data)

        # Run pipeline
        result = pipeline.run()
        result.wait_until_finish()

# Process each file in sequence
for file_name in file_list:
    process_file(file_name)

# Print final results
print("Latest Records:", latest_data_list)
print("Updated Records:", updated_data_list)


Latest Records: [['4', 'Emily', 'Davis', '28', '58000', 'Marketing', '2025-03-10', ''], ['5', 'Daniel', 'Martinez', '40', '90000', 'IT', '2025-03-11', '2025-03-11'], ['6', 'Sophia', 'Lopez', '27', '62000', 'HR', '2025-03-12', ''], ['7', 'James', 'Gonzalez', '45', '110000', 'Finance', '2025-03-13', '2025-03-13'], ['8', 'Olivia', 'Wilson', '32', '70000', 'Marketing', '2025-03-14', ''], ['9', 'William', 'Anderson', '29', '63000', 'IT', '2025-03-15', '2025-03-15'], ['10', 'Ava', 'Thomas', '26', '59000', 'HR', '2025-03-16', '2025-03-16'], ['11', 'Alexander', 'Taylor', '38', '85000', 'Finance', '2025-03-17', ''], ['12', 'Mia', 'Moore', '24', '54000', 'Marketing', '2025-03-18', '2025-03-18'], ['13', 'Ethan', 'Jackson', '31', '67000', 'IT', '2025-03-19', '2025-03-19'], ['14', 'Charlotte', 'White', '33', '72000', 'HR', '2025-03-20', ''], ['15', 'Benjamin', 'Harris', '37', '78000', 'Finance', '2025-03-21', '2025-03-21'], ['16', 'Amelia', 'Clark', '29', '64000', 'Marketing', '2025-03-22', ''], ['

In [ ]:
"""
it merge the outout of the both code
"""


import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions
from datetime import datetime, timedelta

# List of CSV files to process
file_list = ['/content/sample - Sheet1 (1).csv', '/content/sample - Sheet1 (2).csv']

# Define last load date (yesterday)
#LAST_LOAD_DATE = (datetime.today() - timedelta(days=1)).strftime("%Y-%m-%d")
LAST_LOAD_DATE = "2025-03-24"

class ParseCSV(beam.DoFn):
    """Parse CSV lines into dictionaries."""
    def process(self, line, headers):
        values = line.split(',')
        if len(values) == len(headers):  # Ensure proper parsing
            yield dict(zip(headers, values))

class FilterLatest(beam.DoFn):
    """Filter newly inserted records based on 'erdate'."""
    def process(self, record):
        try:
            erdate = datetime.strptime(record['erdate'], "%Y-%m-%d")
            if erdate > datetime.strptime(LAST_LOAD_DATE, "%Y-%m-%d"):
                yield record
        except ValueError:
            pass  # Ignore malformed date entries

class FilterUpdated(beam.DoFn):
    """Filter updated records based on 'udate'."""
    def process(self, record):
        try:
            udate = datetime.strptime(record['udate'], "%Y-%m-%d") if record['udate'] and record['udate'] != "NULL" else None
            if udate and udate > datetime.strptime(LAST_LOAD_DATE, "%Y-%m-%d"):
                yield record
        except ValueError:
            pass  # Ignore malformed date entries

class MergeRecords(beam.DoFn):
    """Merge records by keeping the latest 'udate'."""
    def process(self, group):
        key, records = group
        records = [record for record in records]
        # Sort by udate, keeping latest (or fallback to erdate if missing)
        latest_record = __builtins__.max(records, key=lambda r: r.get('udate', r.get('erdate', '0000-00-00')))
        yield latest_record

# Beam Pipeline
pipeline_options = PipelineOptions()
with beam.Pipeline(options=pipeline_options) as pipeline:

    all_data = []

    for file_name in file_list:
        lines = pipeline | f"Read {file_name}" >> beam.io.ReadFromText(file_name)

        # Extract header separately
        headers = (
            lines
            | f"Extract Header {file_name}" >> beam.transforms.combiners.ToList()
            | f"Get First Row {file_name}" >> beam.Map(lambda rows: rows[0].split(',') if rows else [])
        )

        # Extract data rows (excluding header)
        data_rows = lines | f"Filter Data {file_name}" >> beam.Filter(lambda line: not line.startswith("id"))

        # Convert data rows to dictionaries
        parsed_data = data_rows | f"Parse {file_name}" >> beam.ParDo(ParseCSV(), headers=beam.pvalue.AsSingleton(headers))

        all_data.append(parsed_data)

    # FIXED: Properly merge all PCollections
    merged_data = all_data | "Flatten Data" >> beam.Flatten()

    # Filter latest and updated records
    latest_records = merged_data | "Filter Latest" >> beam.ParDo(FilterLatest())
    updated_records = merged_data | "Filter Updated" >> beam.ParDo(FilterUpdated())

    # Combine latest and updated records
    combined_records = (latest_records, updated_records) | "Merge Incremental Records" >> beam.Flatten()

    # Group by 'id' for merging
    grouped_records = (
        combined_records
        | "Map to (id, record)" >> beam.Map(lambda r: (r['id'], r))
        | "Group By ID" >> beam.GroupByKey()
    )

    # Merge records to keep the latest update
    final_records = grouped_records | "Merge Latest Records" >> beam.ParDo(MergeRecords())

    # Print final merged data (Replace with actual sink)
    updated_records | "Print Final Records" >> beam.Map(print)


{'id': '19', 'first_name': 'Mason', 'last_name': 'Hall', 'age': '34', 'salary': '74000', 'department': 'Finance', 'erdate': '2025-03-25', 'udate': '2025-03-25'}
{'id': '17', 'first_name': 'Lucas', 'last_name': 'Robinson', 'age': '42', 'salary': '98000', 'department': 'IT', 'erdate': '2025-03-23', 'udate': '2025-03-25'}
{'id': '19', 'first_name': 'Mason', 'last_name': 'Hall', 'age': '34', 'salary': '74000', 'department': 'Finance', 'erdate': '2025-03-25', 'udate': '2025-03-25'}


In [ ]:
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions
from datetime import datetime, timedelta

# List of CSV files to process
file_list = ['/content/sample - Sheet1 (1).csv', '/content/sample - Sheet1 (2).csv']

# Define last load date (yesterday)
LAST_LOAD_DATE = (datetime.today() - timedelta(days=1)).strftime("%Y-%m-%d")

class ParseCSV(beam.DoFn):
    """Parse CSV lines into dictionaries."""
    def process(self, line, headers):
        values = line.split(',')
        if len(values) == len(headers):
            yield dict(zip(headers, values))

class FilterLatest(beam.DoFn):
    """Filter newly inserted records based on 'erdate'."""
    def process(self, record):
        try:
            erdate = datetime.strptime(record['erdate'], "%Y-%m-%d")
            if erdate > datetime.strptime(LAST_LOAD_DATE, "%Y-%m-%d"):
                yield record
        except ValueError:
            pass

class FilterUpdated(beam.DoFn):
    """Filter updated records based on 'udate'."""
    def process(self, record):
        try:
            udate = datetime.strptime(record['udate'], "%Y-%m-%d") if record['udate'] and record['udate'] != "NULL" else None
            if udate and udate > datetime.strptime(LAST_LOAD_DATE, "%Y-%m-%d"):
                yield record
        except ValueError:
            pass
class MergeRecords(beam.DoFn):
    """Merge records by keeping the latest 'udate'."""
    def process(self, group):
        key, records_iterable = group  # Ensure proper unpacking
        records = [records_iterable for records_iterable in records_iterable]  # Convert to a list
        if records:
            # Use __builtins__.max to access the built-in max function
            latest_record = __builtins__.max(records, key=lambda r: r.get('udate', r.get('erdate', '0000-00-00')))

            yield latest_record

def process_file(file_name):
    """Process a single file and write output."""
    output_file = file_name.replace('.csv', '_output.csv')

    pipeline_options = PipelineOptions()
    with beam.Pipeline(options=pipeline_options) as pipeline:

        lines = pipeline | f"Read {file_name}" >> beam.io.ReadFromText(file_name)

        # Extract header separately
        headers = (
            lines
            | f"Extract Header {file_name}" >> beam.transforms.combiners.ToList()
            | f"Get First Row {file_name}" >> beam.Map(lambda rows: rows[0].split(',') if rows else [])
        )

        # Extract data rows (excluding header)
        data_rows = lines | f"Filter Data {file_name}" >> beam.Filter(lambda line: not line.startswith("id"))

        # Convert data rows to dictionaries
        parsed_data = data_rows | f"Parse {file_name}" >> beam.ParDo(ParseCSV(), headers=beam.pvalue.AsSingleton(headers))

        # Filter latest and updated records
        latest_records = parsed_data | "Filter Latest" >> beam.ParDo(FilterLatest())
        updated_records = parsed_data | "Filter Updated" >> beam.ParDo(FilterUpdated())

        # Combine latest and updated records
        combined_records = (latest_records, updated_records) | "Merge Incremental Records" >> beam.Flatten()

        # Group by 'id' for merging
        grouped_records = (
            combined_records
            | "Map to (id, record)" >> beam.Map(lambda r: (r['id'], r))
            | "Group By ID" >> beam.GroupByKey()
        )

        # Merge records to keep the latest update
        final_records = grouped_records | "Merge Latest Records" >> beam.ParDo(MergeRecords())

        # Format output as CSV
        formatted_output = final_records | "Format Output" >> beam.Map(lambda r: ','.join(str(v) for v in r.values()))

        # Write results to output CSV file
        formatted_output | "Write to File" >> beam.io.WriteToText(output_file, file_name_suffix=".csv", header="id,first_name,last_name,age,salary,department,erdate,udate")

    print(f" Processed {file_name}, output written to {output_file}")

# Process each file one by one
for file_name in file_list:
    process_file(file_name)


 Processed /content/sample - Sheet1 (1).csv, output written to /content/sample - Sheet1 (1)_output.csv


 Processed /content/sample - Sheet1 (2).csv, output written to /content/sample - Sheet1 (2)_output.csv


In [ ]:
df = spark.read.csv("/content/sample - Sheet1 (1)_output.csv-00000-of-00001.csv", header=True, inferSchema=True)

In [ ]:
df2 =  spark.read.csv("/content/sample - Sheet1 (2)_output.csv-00000-of-00001.csv", header=True, inferSchema=True)

In [ ]:
df.show()

+---+----------+---------+---+------+----------+----------+----------+
| id|first_name|last_name|age|salary|department|    erdate|     udate|
+---+----------+---------+---+------+----------+----------+----------+
|  5|    Daniel| Martinez| 40| 90000|        IT|2025-03-11|2025-03-11|
|  7|     James| Gonzalez| 45|110000|   Finance|2025-03-13|2025-03-13|
|  9|   William| Anderson| 29| 63000|        IT|2025-03-15|2025-03-15|
| 10|       Ava|   Thomas| 26| 59000|        HR|2025-03-16|2025-03-16|
| 12|       Mia|    Moore| 24| 54000| Marketing|2025-03-18|2025-03-18|
| 13|     Ethan|  Jackson| 31| 67000|        IT|2025-03-19|2025-03-19|
| 15|  Benjamin|   Harris| 37| 78000|   Finance|2025-03-21|2025-03-21|
| 17|     Lucas| Robinson| 42| 98000|        IT|2025-03-23|2025-03-23|
| 19|     Mason|     Hall| 34| 74000|   Finance|2025-03-25|2025-03-25|
|  4|     Emily|    Davis| 28| 58000| Marketing|2025-03-10|      NULL|
|  6|    Sophia|    Lopez| 27| 62000|        HR|2025-03-12|      NULL|
|  8| 

In [ ]:
df2.show()

+---+----------+---------+---+------+----------+----------+----------+
| id|first_name|last_name|age|salary|department|    erdate|     udate|
+---+----------+---------+---+------+----------+----------+----------+
|  5|    Daniel| Martinez| 40| 90000|        IT|2025-03-11|2025-03-11|
|  7|     James| Gonzalez| 45|110000|   Finance|2025-03-13|2025-03-13|
|  9|   William| Anderson| 29| 63000|        IT|2025-03-15|2025-03-15|
| 10|       Ava|   Thomas| 26| 59000|        HR|2025-03-16|2025-03-16|
| 12|       Mia|    Moore| 24| 54000| Marketing|2025-03-18|2025-03-18|
| 13|     Ethan|  Jackson| 31| 67000|        IT|2025-03-19|2025-03-19|
| 15|  Benjamin|   Harris| 37| 78000|   Finance|2025-03-21|2025-03-21|
| 17|     Lucas| Robinson| 42| 98000|        IT|2025-03-23|2025-03-25|
| 19|     Mason|     Hall| 34| 74000|   Finance|2025-03-25|2025-03-25|
|  4|     Emily|    Davis| 28| 58000| Marketing|2025-03-10|      NULL|
|  6|    Sophia|    Lopez| 27| 62000|        HR|2025-03-12|      NULL|
|  8| 

In [ ]:
target_df1 = spark.read.csv("/content/sample - Sheet1 (1).csv", header=True, inferSchema=True)

In [ ]:
target_df2 = spark.read.csv("/content/sample - Sheet1 (2).csv", header=True, inferSchema=True)

In [ ]:
target_df1.show()

+---+----------+---------+---+------+----------+----------+----------+
| id|first_name|last_name|age|salary|department|    erdate|     udate|
+---+----------+---------+---+------+----------+----------+----------+
|  1|      John|      Doe| 30| 60000|        IT|2025-03-07|2025-03-07|
|  2|      Jane|    Smith| 25| 55000|        HR|2025-03-08|      NULL|
|  3|   Michael|  Johnson| 35| 75000|   Finance|2025-03-09|2025-03-09|
|  4|     Emily|    Davis| 28| 58000| Marketing|2025-03-10|      NULL|
|  5|    Daniel| Martinez| 40| 90000|        IT|2025-03-11|2025-03-11|
|  6|    Sophia|    Lopez| 27| 62000|        HR|2025-03-12|      NULL|
|  7|     James| Gonzalez| 45|110000|   Finance|2025-03-13|2025-03-13|
|  8|    Olivia|   Wilson| 32| 70000| Marketing|2025-03-14|      NULL|
|  9|   William| Anderson| 29| 63000|        IT|2025-03-15|2025-03-15|
| 10|       Ava|   Thomas| 26| 59000|        HR|2025-03-16|2025-03-16|
| 11| Alexander|   Taylor| 38| 85000|   Finance|2025-03-17|      NULL|
| 12| 

In [ ]:
target_df2.show()

+---+----------+---------+---+------+----------+----------+----------+
| id|first_name|last_name|age|salary|department|    erdate|     udate|
+---+----------+---------+---+------+----------+----------+----------+
|  1|      John|      Doe| 30| 60000|        IT|2025-03-07|2025-03-07|
|  2|      Jane|    Smith| 25| 55000|        HR|2025-03-08|      NULL|
|  3|   Michael|  Johnson| 35| 75000|   Finance|2025-03-09|2025-03-09|
|  4|     Emily|    Davis| 28| 58000| Marketing|2025-03-10|      NULL|
|  5|    Daniel| Martinez| 40| 90000|        IT|2025-03-11|2025-03-11|
|  6|    Sophia|    Lopez| 27| 62000|        HR|2025-03-12|      NULL|
|  7|     James| Gonzalez| 45|110000|   Finance|2025-03-13|2025-03-13|
|  8|    Olivia|   Wilson| 32| 70000| Marketing|2025-03-14|      NULL|
|  9|   William| Anderson| 29| 63000|        IT|2025-03-15|2025-03-15|
| 10|       Ava|   Thomas| 26| 59000|        HR|2025-03-16|2025-03-16|
| 11| Alexander|   Taylor| 38| 85000|   Finance|2025-03-17|      NULL|
| 12| 

In [ ]:
# Perform Full Outer Join
merged_df = target_df.alias("target").join(
    df.alias("source"),
    on="id",
    how="outer"
)


In [ ]:
# Apply Merge Logic
final_df = merged_df.select(
    col("id"),
    when(col("source.first_name").isNotNull(), col("source.first_name")).otherwise(col("target.first_name")).alias("first_name"),
    when(col("source.last_name").isNotNull(), col("source.last_name")).otherwise(col("target.last_name")).alias("last_name"),
    when(col("source.age").isNotNull(), col("source.age")).otherwise(col("target.age")).alias("age"),
    when(col("source.salary").isNotNull(), col("source.salary")).otherwise(col("target.salary")).alias("salary"),
    when(col("source.department").isNotNull(), col("source.department")).otherwise(col("target.department")).alias("department"),
    when(col("source.erdate").isNotNull(), col("source.erdate")).otherwise(col("target.erdate")).alias("erdate"),
    when(col("source.udate").isNotNull(), col("source.udate")).otherwise(col("target.udate")).alias("udate")
)

In [ ]:
final_df.show()

+---+----------+---------+---+------+----------+----------+----------+
| id|first_name|last_name|age|salary|department|    erdate|     udate|
+---+----------+---------+---+------+----------+----------+----------+
|  1|      John|      Doe| 30| 60000|        IT|2025-03-07|2025-03-07|
|  2|      Jane|    Smith| 25| 55000|        HR|2025-03-08|      NULL|
|  3|   Michael|  Johnson| 35| 75000|   Finance|2025-03-09|2025-03-09|
|  4|     Emily|    Davis| 28| 58000| Marketing|2025-03-10|      NULL|
|  5|    Daniel| Martinez| 40| 90000|        IT|2025-03-11|2025-03-11|
|  6|    Sophia|    Lopez| 27| 62000|        HR|2025-03-12|      NULL|
|  7|     James| Gonzalez| 45|110000|   Finance|2025-03-13|2025-03-13|
|  8|    Olivia|   Wilson| 32| 70000| Marketing|2025-03-14|      NULL|
|  9|   William| Anderson| 29| 63000|        IT|2025-03-15|2025-03-15|
| 10|       Ava|   Thomas| 26| 59000|        HR|2025-03-16|2025-03-16|
| 11| Alexander|   Taylor| 38| 85000|   Finance|2025-03-17|      NULL|
| 12| 

In [ ]:
merge_df1 = target_df1.alias("target").join(
    df2.alias("source"),
    on="id",
    how="outer"
)

In [ ]:
# Apply Merge Logic
final_df1 = merge_df1.select(
    col("id"),
    when(col("source.first_name").isNotNull(), col("source.first_name")).otherwise(col("target.first_name")).alias("first_name"),
    when(col("source.last_name").isNotNull(), col("source.last_name")).otherwise(col("target.last_name")).alias("last_name"),
    when(col("source.age").isNotNull(), col("source.age")).otherwise(col("target.age")).alias("age"),
    when(col("source.salary").isNotNull(), col("source.salary")).otherwise(col("target.salary")).alias("salary"),
    when(col("source.department").isNotNull(), col("source.department")).otherwise(col("target.department")).alias("department"),
    when(col("source.erdate").isNotNull(), col("source.erdate")).otherwise(col("target.erdate")).alias("erdate"),
    when(col("source.udate").isNotNull(), col("source.udate")).otherwise(col("target.udate")).alias("udate")
)

In [ ]:
final_df1.show()

+---+----------+---------+---+------+----------+----------+----------+
| id|first_name|last_name|age|salary|department|    erdate|     udate|
+---+----------+---------+---+------+----------+----------+----------+
|  1|      John|      Doe| 30| 60000|        IT|2025-03-07|2025-03-07|
|  2|      Jane|    Smith| 25| 55000|        HR|2025-03-08|      NULL|
|  3|   Michael|  Johnson| 35| 75000|   Finance|2025-03-09|2025-03-09|
|  4|     Emily|    Davis| 28| 58000| Marketing|2025-03-10|      NULL|
|  5|    Daniel| Martinez| 40| 90000|        IT|2025-03-11|2025-03-11|
|  6|    Sophia|    Lopez| 27| 62000|        HR|2025-03-12|      NULL|
|  7|     James| Gonzalez| 45|110000|   Finance|2025-03-13|2025-03-13|
|  8|    Olivia|   Wilson| 32| 70000| Marketing|2025-03-14|      NULL|
|  9|   William| Anderson| 29| 63000|        IT|2025-03-15|2025-03-15|
| 10|       Ava|   Thomas| 26| 59000|        HR|2025-03-16|2025-03-16|
| 11| Alexander|   Taylor| 38| 85000|   Finance|2025-03-17|      NULL|
| 12| 

In [ ]:
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions
from datetime import datetime, timedelta

# List of input CSV files
file_list = ['/content/sample - Sheet1 (1).csv', '/content/sample - Sheet1 (1).csv']

# Define last load date (yesterday)
LAST_LOAD_DATE = (datetime.today() - timedelta(days=1)).strftime("%Y-%m-%d")

class ParseCSV(beam.DoFn):
    """Parse CSV lines into dictionaries."""
    def process(self, line, headers):
        values = line.split(',')
        if len(values) == len(headers):
            yield dict(zip(headers, values))

class FilterLatest(beam.DoFn):
    """Filter newly inserted records based on 'erdate'."""
    def process(self, record):
        try:
            erdate = datetime.strptime(record['erdate'], "%Y-%m-%d")
            if erdate > datetime.strptime(LAST_LOAD_DATE, "%Y-%m-%d"):
                yield record
        except ValueError:
            pass  # Ignore malformed date entries

class FilterUpdated(beam.DoFn):
    """Filter updated records based on 'udate'."""
    def process(self, record):
        try:
            udate = datetime.strptime(record['udate'], "%Y-%m-%d") if record['udate'] and record['udate'] != "NULL" else None
            if udate and udate > datetime.strptime(LAST_LOAD_DATE, "%Y-%m-%d"):
                yield record
        except ValueError:
            pass  # Ignore malformed date entries

def process_file(input_file, pipeline_options):
    """Process a single file, filter records, and write output to a new file."""
    with beam.Pipeline(options=pipeline_options) as pipeline:
        # Read CSV file
        lines = pipeline | f"Read {input_file}" >> beam.io.ReadFromText(input_file, skip_header_lines=1)

        # Extract header
        headers = ['id', 'first_name', 'last_name', 'age', 'salary', 'department', 'erdate', 'udate']

        # Parse data
        parsed_data = lines | f"Parse {input_file}" >> beam.ParDo(ParseCSV(), headers=headers)

        # Filter latest and updated records
        latest_records = parsed_data | f"Filter Latest {input_file}" >> beam.ParDo(FilterLatest())
        updated_records = parsed_data | f"Filter Updated {input_file}" >> beam.ParDo(FilterUpdated())

        # Merge latest and updated records
        combined_records = (latest_records, updated_records) | f"Merge {input_file}" >> beam.Flatten()

        # Write to an output file
        formatted_output = combined_records | f"Format {input_file}" >> beam.Map(lambda record: ','.join(record.values()))
        combined_records | f"Write to " >> beam.io.WriteToText(input_file)

# **Process each file sequentially**
pipeline_options = PipelineOptions()

for i in range(len(file_list)):
    input_file = file_list[i]
    #output_file = f"/content/output_{i+1}"  # Output file dynamically named
    process_file(input_file, pipeline_options)

print("Processing complete. Outputs are stored in new files.")


Processing complete. Outputs are stored in new files.
